In [0]:
%run /Workspace/Users/antoniorad15@gmail.com/ROBOTICS-AI-training-pipeline/pipeline-finetune-gr00t/secrets-template

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_4", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_4", 0o600)

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no shadeform@$BREV_IP << 'EOF'
echo == OS ==
. /etc/os-release
echo $PRETTY_NAME
echo == GPU / Driver ==
nvidia-smi
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no shadeform@$BREV_IP << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin";
curl -LsSf https://astral.sh/uv/install.sh | /bin/sh;
uv --version;
sudo apt-get update
sudo apt-get install -y libglu1-mesa tmux
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no shadeform@$BREV_IP << 'EOF'
git clone https://github.com/REBELDOT-SOLUTIONS-S-R-L/ROBOTICS-lehome-challenge.git
EOF

In [0]:
%sh
# Ctrl+C the current command, then:
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no shadeform@$BREV_IP << 'EOF'
# kill any hanging hf processes
pkill -f "hf download" || true
pkill -f "huggingface" || true

# remove stale lock files
find ~/ROBOTICS-lehome-challenge/Assets/.cache/huggingface/download -name "*.lock" -delete


In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no shadeform@$BREV_IP << 'EOF'
cd ROBOTICS-lehome-challenge
uv sync
git clone -b lehome-cloth-mimic-compat https://github.com/alex-luci/IsaacLab.git third_party/IsaacLab
#test
#git checkout fix/cuda-env-support

git checkout auto-annotation-teleop
git fetch
git pull
git status
#git reset --hard ab8230758ed5ffd2901b9ebc38f0e097e944b537
source .venv/bin/activate
Yes | ./third_party/IsaacLab/isaaclab.sh -i none
uv pip install -e ./source/lehome
hf download lehome/asset_challenge --repo-type dataset --local-dir Assets
EOF
scp -r -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no \
  /Volumes/workspace/default/assets/so101_follower_eef.usd \
  shadeform@$BREV_IP:~/ROBOTICS-lehome-challenge/Assets/robots/lerobot/

In [0]:
# === Pool configuration ===
MAX_CONCURRENT = 3            # max parallel tmux sessions the instance can handle
INPUT_DIR = "~/mimicgen_dataset"  # remote folder containing annotated HDF5 files
SESSION_PREFIX = "generate"
SSH_KEY = "/tmp/ssh_private_key_4"
POLL_INTERVAL = 30            # seconds between status checks

In [ ]:
%sh
# Patch generation_runtime.py on Brev: add None guard around pose_writer.close()
# so the script exits cleanly instead of crashing with AttributeError,
# which lets the tmux teardown chain always reach tmux kill-session.
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no shadeform@$BREV_IP << 'EOF'
FILE=~/ROBOTICS-lehome-challenge/scripts/mimicgen/core/generation_runtime.py
python3 - << 'PYEOF'
import re, pathlib
p = pathlib.Path('/home/shadeform/ROBOTICS-lehome-challenge/scripts/mimicgen/core/generation_runtime.py')
src = p.read_text()
patched = re.sub(
    r'^( +)pose_writer\.close\(\)',
    r'\1if pose_writer is not None:\n\1    pose_writer.close()',
    src,
    flags=re.MULTILINE,
)
if patched == src:
    print('Already patched — no change.')
else:
    p.write_text(patched)
    print('Patched: pose_writer.close() now has None guard.')
PYEOF
grep -n 'pose_writer' $FILE | tail -6
EOF


In [0]:
import subprocess, time, os

BREV_IP = os.environ["BREV_IP"]

def ssh(cmd):
    r = subprocess.run(
        ["ssh", "-i", SSH_KEY, "-o", "StrictHostKeyChecking=no", f"shadeform@{BREV_IP}", cmd],
        capture_output=True, text=True
    )
    return r.stdout.strip(), r.returncode

# --- discover input files ---
out, _ = ssh(f"ls {INPUT_DIR}/*.hdf5 2>/dev/null | sort")
files = [f for f in out.split("\n") if f.strip()]
print(f"Found {len(files)} input files:")
for f in files:
    print(f"  {f}")

def active_sessions():
    out, rc = ssh("tmux list-sessions -F '#{session_name}' 2>/dev/null || true")
    if rc != 0 or not out:
        return set()
    return {s for s in out.split("\n") if s.startswith(SESSION_PREFIX + "_")}

GARMENT_NAMES = ["Top_Long_Seen_0", "Top_Long_Seen_1"]

def launch(filepath, garment_name):
    stem = filepath.split("/")[-1].replace(".hdf5", "")
    sname = f"{SESSION_PREFIX}_{garment_name}"
    out_stem = f"generated_dataset_{garment_name}"

    # Teardown chain:
    #   1. SIGTERM kit — gives Isaac Sim a chance to flush & exit cleanly
    #   2. sleep 10    — wait for graceful shutdown
    #   3. SIGKILL kit — force-kill anything still alive
    #   4. kill-session — drop the tmux window
    teardown = (
        "pkill -f 'generate_dataset.py' 2>/dev/null || true ; "
        "pkill -TERM -f 'kit' 2>/dev/null || true ; "
        "sleep 10 ; "
        "pkill -KILL -f 'kit' 2>/dev/null || true ; "
        f"tmux kill-session -t {sname}"
    )

    cmd = f"""
tmux kill-session -t {sname} 2>/dev/null || true
sleep 2
tmux new-session -d -s {sname}
tmux send-keys -t {sname} 'cd ROBOTICS-lehome-challenge && source .venv/bin/activate && yes Yes | timeout 7200 python scripts/mimicgen/generate_dataset.py \
  --task LeHome-BiSO101-ManagerBased-Garment-Mimic-v0 \
  --garment_name {garment_name} \
  --input_file {filepath} \
  --output_file Datasets/hdf5_datasets/4_generated_datasets/{out_stem}.hdf5 \
  --generation_num_trials 1 \
  --device cuda \
  --num_envs 1 \
  --enable_cameras \
  --logging_interval 10 \
  --log_success \
  --headless ; {teardown}' Enter
"""
    ssh(cmd)
    print(f"  [STARTED] {sname}  ({filepath}, {garment_name})")
    return sname

# --- build queue as (file, garment) pairs ---
queue = [(f, g) for f in files for g in GARMENT_NAMES]

launched = set()
completed = set()

print(f"\nStarting pool (MAX_CONCURRENT={MAX_CONCURRENT}, {len(queue)} jobs)...")

while queue and len(launched - completed) < MAX_CONCURRENT:
    f, g = queue.pop(0)
    launched.add(launch(f, g))

while launched - completed:
    time.sleep(POLL_INTERVAL)
    active = active_sessions()
    for s in list(launched - completed - active):
        print(f"  [DONE] {s}")
        completed.add(s)
    while queue and len(launched - completed) < MAX_CONCURRENT:
        f, g = queue.pop(0)
        launched.add(launch(f, g))
    print(f"  active={len(launched - completed)}  done={len(completed)}  queued={len(queue)}")

print(f"\nAll {len(completed)} sessions completed.")

In [0]:
# import subprocess, time, os

# BREV_IP = os.environ["BREV_IP"]

# def ssh(cmd):
#     r = subprocess.run(
#         ["ssh", "-i", SSH_KEY, "-o", "StrictHostKeyChecking=no", f"shadeform@{BREV_IP}", cmd],
#         capture_output=True, text=True
#     )
#     return r.stdout.strip(), r.returncode

# # --- discover input files ---
# out, _ = ssh(f"ls {INPUT_DIR}/*.hdf5 2>/dev/null | sort")
# files = [f for f in out.split("\n") if f.strip()]
# print(f"Found {len(files)} input files:")
# for f in files:
#     print(f"  {f}")

# def active_sessions():
#     out, rc = ssh("tmux list-sessions -F '#{session_name}' 2>/dev/null || true")
#     if rc != 0 or not out:
#         return set()
#     return {s for s in out.split("\n") if s.startswith(SESSION_PREFIX + "_")}

# GARMENT_NAMES = ["Top_Long_Seen_0", "Top_Long_Seen_1"]

# def launch(filepath, garment_name):
#     stem = filepath.split("/")[-1].replace(".hdf5", "")
#     sname = f"{SESSION_PREFIX}_{garment_name}"
#     out_stem = f"generated_dataset_{garment_name}"

#     cmd = f"""
# tmux kill-session -t {sname} 2>/dev/null || true
# sleep 2
# tmux new-session -d -s {sname}
# tmux send-keys -t {sname} 'cd ROBOTICS-lehome-challenge && source .venv/bin/activate && yes Yes | timeout 7200 python scripts/mimicgen/generate_dataset.py \
#   --task LeHome-BiSO101-ManagerBased-Garment-Mimic-v0 \
#   --garment_name {garment_name} \
#   --input_file {filepath} \
#   --output_file Datasets/hdf5_datasets/4_generated_datasets/{out_stem}.hdf5 \
#   --generation_num_trials 1 \
#   --device cuda \
#   --num_envs 1 \
#   --enable_cameras \
#   --logging_interval 10 \
#   --log_success \
#   --headless ; pkill -f "generate_dataset.py" 2>/dev/null; pkill -f "kit" 2>/dev/null; sleep 5 ; tmux kill-session -t {sname}' Enter
# """
#     ssh(cmd)
#     print(f"  [STARTED] {sname}  ({filepath}, {garment_name})")
#     return sname

# # --- build queue as (file, garment) pairs ---
# queue = [(f, g) for f in files for g in GARMENT_NAMES]

# launched = set()
# completed = set()

# print(f"\nStarting pool (MAX_CONCURRENT={MAX_CONCURRENT}, {len(queue)} jobs)...")

# while queue and len(launched - completed) < MAX_CONCURRENT:
#     f, g = queue.pop(0)
#     launched.add(launch(f, g))

# while launched - completed:
#     time.sleep(POLL_INTERVAL)
#     active = active_sessions()
#     for s in list(launched - completed - active):
#         print(f"  [DONE] {s}")
#         completed.add(s)
#     while queue and len(launched - completed) < MAX_CONCURRENT:
#         f, g = queue.pop(0)
#         launched.add(launch(f, g))
#     print(f"  active={len(launched - completed)}  done={len(completed)}  queued={len(queue)}")

# print(f"\nAll {len(completed)} sessions completed.")

In [0]:
import subprocess, time, os

BREV_IP = os.environ["BREV_IP"]
SSH_KEY = "/tmp/ssh_private_key_4"
INPUT_CONVERT_DIR = "~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/4_generated_datasets"
OUTPUT_DIR = "~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/5_final_generated_datasets"
CONVERT_PREFIX = "convert"
MAX_CONCURRENT_CONVERT = 6
POLL_INTERVAL_CONVERT = 15

def ssh(cmd):
    r = subprocess.run(
        ["ssh", "-i", SSH_KEY, "-o", "StrictHostKeyChecking=no", f"shadeform@{BREV_IP}", cmd],
        capture_output=True, text=True
    )
    return r.stdout.strip(), r.returncode

# ensure output directory exists
ssh(f"mkdir -p {OUTPUT_DIR}")
ssh("mkdir -p ~/convert_logs")

# kill any lingering writers from the generation step that might still hold
# HDF5 file locks on the inputs, otherwise the converter hits EAGAIN on open()
ssh("pkill -f 'generate_dataset.py' 2>/dev/null || true")
ssh("pkill -f 'kit' 2>/dev/null || true")
time.sleep(3)

# discover generated files (exclude _failed)
out, _ = ssh(f"ls {INPUT_CONVERT_DIR}/*.hdf5 2>/dev/null | grep -v '_failed' | sort")
files = [f for f in out.split("\n") if f.strip()]
print(f"Found {len(files)} files to convert:")
for f in files:
    print(f"  {f}")

def active_sessions():
    out, rc = ssh("tmux list-sessions -F '#{session_name}' 2>/dev/null || true")
    if rc != 0 or not out:
        return set()
    return {s for s in out.split("\n") if s.startswith(CONVERT_PREFIX + "_")}

def launch_convert(filepath):
    stem = filepath.split("/")[-1].replace(".hdf5", "")
    out_stem = f"joint_{stem}"
    sname = f"{CONVERT_PREFIX}_{stem}"
    log_path = f"~/convert_logs/{stem}.log"
    # HDF5_USE_FILE_LOCKING=FALSE tells libhdf5 to skip flock() on open.
    # Needed because (a) the generation step may leave stale locks after pkill
    # and (b) some Brev filesystems return EAGAIN on flock() even when unused.
    cmd = f"""
tmux kill-session -t {sname} 2>/dev/null || true
tmux new-session -d -s {sname}
tmux send-keys -t {sname} 'cd ROBOTICS-lehome-challenge && source .venv/bin/activate && export HDF5_USE_FILE_LOCKING=FALSE && python scripts/mimicgen/leisaac_eef_action_process.py \
  --input_file {filepath} \
  --output_file {OUTPUT_DIR}/{out_stem}.hdf5 \
  --to_joint \
  --headless > {log_path} 2>&1 ; tmux kill-session -t {sname}' Enter
"""
    ssh(cmd)
    print(f"  [STARTED] {sname}  (log: {log_path})")
    return sname

queue = list(files)
launched = set()
completed = set()

while queue and len(launched - completed) < MAX_CONCURRENT_CONVERT:
    launched.add(launch_convert(queue.pop(0)))

while launched - completed:
    time.sleep(POLL_INTERVAL_CONVERT)
    active = active_sessions()
    for s in list(launched - completed - active):
        print(f"  [DONE] {s}")
        completed.add(s)
    while queue and len(launched - completed) < MAX_CONCURRENT_CONVERT:
        launched.add(launch_convert(queue.pop(0)))
    print(f"  active={len(launched - completed)}  done={len(completed)}  queued={len(queue)}")

print(f"\nAll {len(completed)} conversions completed.")

# print logs for inspection
import time as _t
_t.sleep(2)
for f in files:
    stem = f.split("/")[-1].replace(".hdf5", "")
    log, _ = ssh(f"cat ~/convert_logs/{stem}.log 2>/dev/null | tail -30")
    print(f"\n=== Log: {stem} ===")
    print(log or "(empty)")

In [0]:
import subprocess, os

BREV_IP = os.environ["BREV_IP"]
SSH_KEY = "/tmp/ssh_private_key_4"
INPUT_MERGE_DIR = "~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/5_final_generated_datasets"
OUTPUT_MERGED = "~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/6_merged_datasets/merged.hdf5"

def ssh(cmd):
    r = subprocess.run(
        ["ssh", "-i", SSH_KEY, "-o", "StrictHostKeyChecking=no", f"shadeform@{BREV_IP}", cmd],
        capture_output=True, text=True
    )
    return r.stdout.strip(), r.returncode

# Ensure output directory exists
ssh(f"mkdir -p ~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/6_merged_datasets")

# Discover all converted HDF5 files
out, _ = ssh(f"ls {INPUT_MERGE_DIR}/*.hdf5 2>/dev/null | sort")
files = [f for f in out.split("\n") if f.strip()]
print(f"Found {len(files)} files to merge:")
for f in files:
    print(f"  {f}")

if not files:
    raise RuntimeError("No HDF5 files found in 5_final_generated_datasets — run the conversion step first.")

# Build and run merge command — kill the session when done so the polling loop exits
file_args = " ".join(files)
merge_cmd = (
    f"cd ROBOTICS-lehome-challenge && source .venv/bin/activate && "
    f"python scripts/mimicgen/merge_hdf5.py {file_args} --output {OUTPUT_MERGED} "
    f"&& echo MERGE_DONE || echo MERGE_FAILED ; tmux kill-session -t merge_datasets"
)

ssh("tmux kill-session -t merge_datasets 2>/dev/null || true")
ssh(f"tmux new-session -d -s merge_datasets")
ssh(f"tmux send-keys -t merge_datasets '{merge_cmd}' Enter")
print("Merge started in tmux session 'merge_datasets'. Waiting for completion...")

import time
while True:
    time.sleep(20)
    active, _ = ssh("tmux list-sessions -F '#{session_name}' 2>/dev/null || true")
    if "merge_datasets" not in active.split("\n"):
        print("Tmux session ended.")
        break
    print("  still merging...")

# Check output
_, rc = ssh(f"test -f {OUTPUT_MERGED}")
if rc == 0:
    size, _ = ssh(f"du -sh {OUTPUT_MERGED}")
    print(f"\nMerge complete: {OUTPUT_MERGED}  ({size})")
else:
    print("\nERROR: merged.hdf5 not found — check tmux logs with: tmux capture-pane -t merge_datasets -p")

In [0]:
# %sh
# rsync -avz --progress \
#   -e "ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no" \
#   ubuntu@$BREV_IP:~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/4_generated_datasets/*.hdf5 \
#   /Volumes/workspace/default/hdf5datasets_lehome_many_clothes/

In [0]:
%sh
rsync -avz --progress \
  -e "ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no" \
  shadeform@$BREV_IP:~/ROBOTICS-lehome-challenge/*.mp4 \
  /Volumes/workspace/default/hdf5datasets_lehome_many_clothes/

In [0]:
%sh
rsync -avz --progress \
  -e "ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no" \
  shadeform@$BREV_IP:~/ROBOTICS-lehome-challenge/Datasets/hdf5_datasets/6_merged_datasets/merged.hdf5 \
  /Volumes/workspace/default/hdf5_test/